In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import train_test_split

np.random.seed(42)
N = 400
# "global feature" ve "local feature" yerine geçen sentetik veriler
global_feat = np.random.randn(N, 4)
local_feat  = np.random.randn(N, 4)
y = ((global_feat[:,0] + local_feat[:,1]) > 0).astype(int)

X0 = np.hstack([global_feat, local_feat])   # X0 = [global || local]
X_train, X_val, y_train, y_val, g_train, g_val, l_train, l_val = train_test_split(
    X0, y, global_feat, local_feat, test_size=0.3, random_state=42)

print("X0 shape:", X0.shape)

X0 shape: (400, 8)


In [ ]:
def train_layer(X, y, n_estimators=100, random_state=0):
    learners = [
        RandomForestClassifier(n_estimators=n_estimators, random_state=random_state),
        RandomForestClassifier(n_estimators=n_estimators, random_state=random_state+1),
        ExtraTreesClassifier(n_estimators=n_estimators, random_state=random_state),
        ExtraTreesClassifier(n_estimators=n_estimators, random_state=random_state+1),
    ]
    for m in learners:
        m.fit(X, y)
    return learners

def layer_probs(learners, X):
    # her öğrenici P(y=1) olasılığını verir, iki sınıf için [p0,p1] alıyoruz
    probs = [m.predict_proba(X) for m in learners]
    return np.hstack(probs)  # shape: (N, 4*|y|)

layer0 = train_layer(X_train, y_train)
P0_train = layer_probs(layer0, X_train)
P0_val   = layer_probs(layer0, X_val)
print("P0_train shape:", P0_train.shape)

P0_train shape: (280, 8)


In [ ]:
def val_accuracy(P, y_true, n_learners=4, n_classes=2):
    P_reshaped = P.reshape(P.shape[0], n_learners, n_classes)
    avg_probs = P_reshaped.mean(axis=1)
    preds = avg_probs.argmax(axis=1)
    return (preds == y_true).mean()

acc0 = val_accuracy(P0_val, y_val)
print("Layer 0 val accuracy:", acc0)

Layer 0 val accuracy: 0.95


In [ ]:
X1_train = np.hstack([P0_train, g_train, l_train])
X1_val   = np.hstack([layer_probs(layer0, X_val), g_val, l_val])

layer1 = train_layer(X1_train, y_train, random_state=10)
P1_val = layer_probs(layer1, X1_val)
acc1 = val_accuracy(P1_val, y_val)
print("X1 shape:", X1_train.shape, " | Layer 1 val accuracy:", acc1)

X1 shape: (280, 16)  | Layer 1 val accuracy: 0.95


In [ ]:
accs = [acc0, acc1]
# ... (döngü halinde daha fazla katman eklenip accs listesine eklenebilir)
def check_stop(accs, l_max=10):
    l = len(accs) - 1
    if l+1 >= l_max:
        return l_max, "Lmax'a ulaşıldı"
    if l >= 2 and accs[l] <= accs[l-1] and accs[l] <= accs[l-2]:
        return l-1, "2 katman üst üste iyileşme yok"
    return None, "devam"

print(check_stop(accs))

(None, 'devam')


Beklenen çıktı: (None, 'devam') çünkü henüz sadece 2 katmanımız var (koşul en az 3 katman gerektirir). Gerçek uygulamada bu döngü, koşul sağlanana kadar devam eder.